# Phase 2: Domain-Informed Feature Engineering

we are applying deep financial domain knowledge instead of blind statistical aggregations. we will calculate **Net Cash Flow**, **Financial Runway**, **Shock Indicators**, and **Volatility** to accurately model Liquidity Stress.

In [4]:
import pandas as pd
import numpy as np
import os

# Load cleaned data
train_df = pd.read_parquet('../data/cleaned/train_clean.parquet', engine='fastparquet')
test_df = pd.read_parquet('../data/cleaned/test_clean.parquet', engine='fastparquet')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (40000, 191)
Test shape: (30000, 190)


## 1. Domain Feature Extraction Logic
We categorize transactions into `Inflows` and `Outflows` to build our custom financial metrics.

### [NN-INSPIRED FEATURE ADDITION]
We are taking the feature transformations that worked best for the Deep Learning models (`signed_log1p` and `zero_inflation_flags`) and injecting them into the Traditional Tree pipeline. 
This explicitly helps Tree models deal with extreme right-tail skewness and hard zeros without wasting tree splits.

In [5]:
def extract_financial_features(df):
    df = df.copy()
    eps = 1e-5 # Prevent division by zero
    
    inflow_tx = ['deposit', 'transfer_from_bank', 'received']
    outflow_tx = ['withdraw', 'paybill', 'merchantpay', 'mm_send']
    
    # --- 1. Monthly Net Cash Flow ---
    for month in range(1, 7):
        in_cols = [f'm{month}_{tx}_total_value' for tx in inflow_tx if f'm{month}_{tx}_total_value' in df.columns]
        out_cols = [f'm{month}_{tx}_total_value' for tx in outflow_tx if f'm{month}_{tx}_total_value' in df.columns]
        
        df[f'm{month}_total_inflow'] = df[in_cols].sum(axis=1) if in_cols else 0
        df[f'm{month}_total_outflow'] = df[out_cols].sum(axis=1) if out_cols else 0
        df[f'm{month}_net_cash_flow'] = df[f'm{month}_total_inflow'] - df[f'm{month}_total_outflow']

    # --- 2. Global Cash Flow & Liquidity Ratios ---
    total_inflow_cols = [f'm{i}_total_inflow' for i in range(1, 7)]
    total_outflow_cols = [f'm{i}_total_outflow' for i in range(1, 7)]
    
    df['total_6m_inflow'] = df[total_inflow_cols].sum(axis=1)
    df['total_6m_outflow'] = df[total_outflow_cols].sum(axis=1)
    df['net_cash_flow_6m'] = df['total_6m_inflow'] - df['total_6m_outflow']
    df['inflow_outflow_ratio'] = df['total_6m_inflow'] / (df['total_6m_outflow'] + eps)

    # --- 3. Financial Runway (Survival Time) ---
    # If they stop making money, how long can their M6 balance sustain their average spending?
    df['avg_monthly_outflow'] = df['total_6m_outflow'] / 6.0
    if 'm6_daily_avg_bal' in df.columns:
        df['runway_months'] = df['m6_daily_avg_bal'] / (df['avg_monthly_outflow'] + eps)
        
    # --- 4. Recent Shock Indicators (M6 vs Baseline) ---
    if 'm6_total_inflow' in df.columns:
        # Did their income suddenly crash in the last month?
        df['baseline_inflow'] = df[[f'm{i}_total_inflow' for i in range(1, 6)]].mean(axis=1)
        df['inflow_shock'] = df['m6_total_inflow'] / (df['baseline_inflow'] + eps)
        
        # Did their expenses suddenly spike in the last month?
        df['baseline_outflow'] = df[[f'm{i}_total_outflow' for i in range(1, 6)]].mean(axis=1)
        df['outflow_shock'] = df['m6_total_outflow'] / (df['baseline_outflow'] + eps)

    # --- 5. Balance Depletion Rate ---
    # How much of their peak safety net is left?
    bal_cols = [f'm{i}_daily_avg_bal' for i in range(1, 7) if f'm{i}_daily_avg_bal' in df.columns]
    if len(bal_cols) == 6:
        df['peak_balance'] = df[bal_cols].max(axis=1)
        df['depletion_rate'] = df['m6_daily_avg_bal'] / (df['peak_balance'] + eps)
        
    # --- 6. Expense Volatility ---
    # Erratic spending is a massive predictor of financial instability.
    df['outflow_std'] = df[total_outflow_cols].std(axis=1).fillna(0)
    df['outflow_volatility'] = df['outflow_std'] / (df['avg_monthly_outflow'] + eps)
    
    # --- 7. Basic Aggregations (we will keep some of the old math just in case) ---
    tx_types = ['paybill', 'merchantpay', 'transfer_from_bank', 'mm_send', 'received', 'deposit', 'withdraw']
    for tx in tx_types:
        cols = [f'm{i}_{tx}_volume' for i in range(1, 7) if f'm{i}_{tx}_volume' in df.columns]
        if cols:
            df[f'{tx}_volume_mean'] = df[cols].mean(axis=1)
            
    # --- [NN-INSPIRED FEATURE ADDITION] ---
    # 8. Zero-Inflation Flags
    # Trees sometimes struggle to separate "exactly zero" from "very small". Explicit flags help.
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if col != "liquidity_stress_next_30d":
            df[f"{col}_is_zero"] = (df[col] == 0).astype(int)
            
    # 9. Signed Log1p Transformation
    # Squashes massive outliers in financial columns so Trees don't overfit to billionaires
    def signed_log1p(x):
        return np.sign(x) * np.log1p(np.abs(x))
        
    skewed_kws = ["amount", "value", "volume", "bal", "flow", "runway", "peak", "shock", "volatility"]
    skewed_cols = [c for c in num_cols if any(kw in c for kw in skewed_kws) and c != "liquidity_stress_next_30d"]
    for col in skewed_cols:
        df[col] = signed_log1p(df[col])
    # --------------------------------------

    return df

## 2. Apply and Save

In [6]:
print("Extracting domain features for Train...")
train_feat = extract_financial_features(train_df)

print("Extracting domain features for Test...")
test_feat = extract_financial_features(test_df)

print(f"\nNew Train shape: {train_feat.shape}")
print(f"New Test shape: {test_feat.shape}")

# Ensure the directory exists
os.makedirs('../data/features', exist_ok=True)

print("Saving Domain-Informed features to parquet...")
train_feat.to_parquet('../data/features/train_features.parquet', index=False, engine='fastparquet')
test_feat.to_parquet('../data/features/test_features.parquet', index=False, engine='fastparquet')
print("Successfully saved!")

Extracting domain features for Train...


/var/folders/c8/7krwwn3d4fq45_lf7twc4d7w0000gn/T/ipykernel_86444/2848579415.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_is_zero"] = (df[col] == 0).astype(int)
/var/folders/c8/7krwwn3d4fq45_lf7twc4d7w0000gn/T/ipykernel_86444/2848579415.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_is_zero"] = (df[col] == 0).astype(int)
/var/folders/c8/7krwwn3d4fq45_lf7twc4d7w0000gn/T/ipykernel_86444/2848579415.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

Extracting domain features for Test...


/var/folders/c8/7krwwn3d4fq45_lf7twc4d7w0000gn/T/ipykernel_86444/2848579415.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_is_zero"] = (df[col] == 0).astype(int)
/var/folders/c8/7krwwn3d4fq45_lf7twc4d7w0000gn/T/ipykernel_86444/2848579415.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_is_zero"] = (df[col] == 0).astype(int)
/var/folders/c8/7krwwn3d4fq45_lf7twc4d7w0000gn/T/ipykernel_86444/2848579415.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f


New Train shape: (40000, 446)
New Test shape: (30000, 445)
Saving Domain-Informed features to parquet...
Successfully saved!
